# Week 4: Math ปิดท้าย + Data Preparation
### คอร์ส Machine Learning & AI 2026
---

**สิ่งที่จะได้เรียนวันนี้:**

1. **ทบทวน Week 3 + ปิดท้าย Math for ML** (30%) — เก็ทว่า gradient มาจากไหน แบบเห็นภาพ
2. **Data Cleaning** — จัดการข้อมูลหาย, ซ้ำ, outlier
3. **Feature Engineering** — สร้าง feature ใหม่ + One-Hot + Scaling
4. **กับดัก ML** — Bias, Noise, **Data Leakage** (สำคัญมาก!)
5. **Mini Project** — สร้าง Data Prep Pipeline ครบวงจร (พร้อมไปต่อ Week 5)

**ธีมวันนี้: ค่าน้ำมัน**

ช่วงนี้ราคาน้ำมันผันผวนมาก — เราจะเอาข้อมูลจริงๆ ของบริษัทขนส่งมาทำให้สะอาด แล้วเตรียมให้ AI ใช้ทำนายค่าใช้จ่ายน้ำมัน

**วิธีใช้ Notebook นี้:**
- กด `Shift + Enter` เพื่อรันโค้ดแต่ละ cell
- อ่าน comment (บรรทัดที่ขึ้นต้นด้วย #) ก่อนรันเสมอ
- ช่วง Workshop: ลองเขียนเองก่อน แล้วค่อยดูเฉลยใน cell ถัดไป

### ขั้นตอนแรก: ติดตั้ง Font ภาษาไทยสำหรับ Matplotlib

รัน cell ด้านล่างนี้ **ก่อน** cell อื่นทั้งหมด (รันครั้งเดียวพอ)

In [ ]:
# ===========================================
# ตั้งค่าฟอนต์ภาษาไทยสำหรับ Matplotlib
# รัน cell นี้ก่อนเสมอ (รันครั้งเดียวพอ)
# ===========================================

import matplotlib.pyplot as plt
import matplotlib

# ดาวน์โหลดและติดตั้งฟอนต์ Sarabun
!wget -q https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf -O /usr/share/fonts/Sarabun-Regular.ttf
!fc-cache -f

matplotlib.font_manager.fontManager.addfont('/usr/share/fonts/Sarabun-Regular.ttf')
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False

print("ติดตั้งฟอนต์ภาษาไทยสำเร็จ!")

In [ ]:
# ===========================================
# Import Libraries หลัก
# ===========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ตั้ง random seed ให้ผลออกมาเหมือนกันทุกคน
np.random.seed(42)

print("Import libraries สำเร็จ!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---
## ช่วงที่ 1: ทบทวน Week 3 + ปิดท้าย Math for ML
---

ช่วงนี้เราจะ:
1. ทบทวนสิ่งที่เรียนไปใน Week 3 อย่างรวดเร็ว
2. ตอบคำถามค้างคาใจ: **"สูตร gradient มาจากไหน?"** (แบบเห็นภาพ ไม่ต้องคำนวณ)
3. ดู Loss Landscape แบบ 3D — เห็น AI เดินหาคำตอบจริงๆ
4. Matrix Operations — เตรียมสู่ Neural Network

### 1.1 ทบทวน Week 3 — Gradient Descent คืออะไร

จาก Week 3 เราเห็นแล้วว่า ML ก็คือการ **"หาเส้นที่ fit กับข้อมูล"**:

```
1. เดาเส้นแรก (สุ่ม)           y = a * x + b
2. ดูว่าผิดเท่าไหร่             Loss = MSE
3. ปรับทีละนิดให้ผิดน้อยลง       a, b ใหม่
4. ทำซ้ำจนได้เส้นที่ดีที่สุด
```

วิธีที่ใช้ปรับค่าเรียกว่า **Gradient Descent** — เหมือนเดินลงเขาตอนปิดตา

In [ ]:
# --- ทบทวน: Gradient Descent จาก Week 3 ---
# ข้อมูลตัวอย่างง่าย ๆ: y = 2x
X = np.array([1, 2, 3, 4, 5], dtype=float)
Y = np.array([2, 4, 6, 8, 10], dtype=float)

# เริ่มต้น
a = 0.0
learning_rate = 0.04
epochs = 20

print(f"เริ่มต้น: a = {a:.4f}")
print("-" * 40)

for epoch in range(epochs):
    y_pred = a * X
    loss = np.mean((Y - y_pred) ** 2)

    # สูตรที่ Week 3 ใช้แต่ไม่ได้อธิบายที่มา
    gradient = -2 * np.mean(X * (Y - y_pred))

    a = a - learning_rate * gradient

    if epoch % 4 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1:2d}: a = {a:.4f}, Loss = {loss:.4f}")

print("-" * 40)
print(f"ค่า a สุดท้าย = {a:.4f}  (ใกล้ 2 มาก เพราะความจริงคือ y = 2x)")

### 1.2 คำถามที่ค้างมาจาก Week 3: "สูตร gradient มาจากไหน?"

ใน Week 3 เราใช้สูตรนี้:

```python
gradient = -2 * np.mean(X * (Y - y_pred))
```

แต่ **ที่มาจริง ๆ คือ "Derivative" (อนุพันธ์)** — ซึ่งเป็นคำที่ฟังดูยาก แต่ความหมายง่ายมาก:

> **Derivative = ความชันของเส้นกราฟ ณ จุดหนึ่ง**

เราจะดูเป็นรูปกันครับ ไม่ต้องคำนวณเอง

In [ ]:
# --- Derivative = ความชัน ---
# ยกตัวอย่าง: f(x) = x²  (เป็นกราฟรูปชาม U)

x = np.linspace(-3, 3, 100)
y = x ** 2

# จุดที่เราจะดูความชัน
points = [-2, 0, 1.5]
colors_pt = ['#e74c3c', '#2ecc71', '#3498db']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, pt, c in zip(axes, points, colors_pt):
    # กราฟหลัก
    ax.plot(x, y, 'k-', linewidth=2, label='f(x) = x²')

    # จุด
    ax.scatter([pt], [pt**2], color=c, s=150, zorder=5)

    # เส้นสัมผัส (tangent line) = ความชัน ณ จุดนั้น
    slope = 2 * pt  # นี่คือ derivative ของ x² = 2x
    x_line = np.linspace(pt - 1.2, pt + 1.2, 20)
    y_line = slope * (x_line - pt) + pt ** 2
    ax.plot(x_line, y_line, color=c, linewidth=3, alpha=0.8,
            label=f'ความชัน = {slope:.1f}')

    ax.set_title(f'ที่จุด x = {pt}', fontsize=13)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.legend(loc='upper center')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-2, 10)

plt.suptitle('Derivative = ความชันของเส้น ณ จุดต่าง ๆ', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("สังเกต:")
print("  - ความชันเป็นลบ -> กราฟกำลังลง  (ถ้าเดินทางขวา จะลง)")
print("  - ความชัน = 0   -> จุดต่ำสุด!  (ไม่ลง ไม่ขึ้น = เจอคำตอบ)")
print("  - ความชันเป็นบวก -> กราฟกำลังขึ้น (ถ้าเดินทางขวา จะขึ้น)")

**เห็นแล้วใช่ไหมครับ?** Gradient Descent แค่ใช้ความชันบอกว่า:
- ถ้า**ชันลง** (ค่าเป็นลบ) → ไปทางขวา
- ถ้า**ชันขึ้น** (ค่าเป็นบวก) → ไปทางซ้าย
- ถ้า**ชันเป็น 0** → เจอจุดต่ำสุดแล้ว หยุดได้!

สูตร update คือ:
```
ค่าใหม่ = ค่าเก่า - learning_rate × ความชัน
```

**เครื่องหมายลบ** คือเหตุผลที่ gradient descent "ลง" เขาตลอด — ถ้าชันลง ค่าเพิ่ม, ถ้าชันขึ้น ค่าลด

### 1.3 ให้ Python หา Derivative ให้เลย (SymPy)

ไม่ต้องจำสูตร ไม่ต้องคำนวณเอง — **SymPy** ทำให้แทนได้ (เครื่องมือใหม่ปี 2026 ที่ควรรู้จัก!)

In [ ]:
# --- SymPy: ให้ Python คำนวณ derivative ให้ ---
import sympy as sp

# 1. ประกาศตัวแปร
x = sp.Symbol('x')

# 2. เขียนฟังก์ชัน
f = x**2
print(f"ฟังก์ชัน: f(x) = {f}")

# 3. หา derivative
f_prime = sp.diff(f, x)
print(f"Derivative: f'(x) = {f_prime}")
print()

# ลองอีก 2 ฟังก์ชัน
for func in [3*x**2 + 2*x + 1, x**3 - 4*x]:
    deriv = sp.diff(func, x)
    print(f"f(x) = {func}")
    print(f"f'(x) = {deriv}")
    print()

**ประโยชน์:** ถ้าเจอฟังก์ชันซับซ้อน เราแค่เขียน Python แล้วให้ SymPy หาให้ ไม่ต้องเปิดตำราคณิตศาสตร์

### 1.4 Loss Landscape — เห็น AI เดินหาคำตอบจริง ๆ

ใน Week 3 เราหาแค่ค่า `a` ตัวเดียว — Loss เป็นกราฟ **เส้น**

แต่ตอนที่มี 2 ตัวแปร (`a` และ `b`) — Loss กลายเป็น **หุบเขา 3 มิติ**

Gradient Descent คือ "การเดินลงหุบเขา" จริง ๆ

In [ ]:
# --- สร้างข้อมูลสำหรับ plot Loss Landscape ---
# ใช้ข้อมูลง่ายๆ: y = 2x + 1
X_data = np.array([1, 2, 3, 4, 5], dtype=float)
Y_data = np.array([3, 5, 7, 9, 11], dtype=float)

# สร้างช่วงของค่า a และ b ที่จะลอง
a_range = np.linspace(0, 4, 50)
b_range = np.linspace(-1, 3, 50)
A, B = np.meshgrid(a_range, b_range)

# คำนวณ Loss สำหรับทุกคู่ (a, b)
Loss = np.zeros_like(A)
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        y_pred = A[i, j] * X_data + B[i, j]
        Loss[i, j] = np.mean((Y_data - y_pred) ** 2)

# --- Plot 3D + Contour ---
fig = plt.figure(figsize=(14, 5))

# 3D surface
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(A, B, Loss, cmap='viridis', alpha=0.8)
ax1.set_xlabel('a (ความชัน)')
ax1.set_ylabel('b (จุดตัดแกน y)')
ax1.set_zlabel('Loss')
ax1.set_title('Loss Landscape แบบ 3D\n(หุบเขาที่ AI ต้องเดินลง)', fontsize=12)
ax1.view_init(elev=25, azim=45)

# Contour plot (มองจากด้านบน)
ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contour(A, B, Loss, levels=20, cmap='viridis')
ax2.contourf(A, B, Loss, levels=20, cmap='viridis', alpha=0.6)
# จุด minimum จริง (a=2, b=1)
ax2.scatter([2], [1], color='red', s=200, marker='*', zorder=5,
            label='จุดต่ำสุดจริง (a=2, b=1)')
ax2.set_xlabel('a (ความชัน)')
ax2.set_ylabel('b (จุดตัดแกน y)')
ax2.set_title('Contour Plot (มองจากด้านบน)\nเส้นคือระดับความสูงของหุบเขา', fontsize=12)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.colorbar(contour, ax=ax2)
plt.tight_layout()
plt.show()

print("ซ้าย  = มองในมุม 3 มิติ (เห็นเป็นหุบเขา)")
print("ขวา   = มองจากด้านบน (เห็นเป็นเส้นระดับ เหมือนแผนที่)")
print("ดาวแดง = จุดต่ำสุด คือคำตอบที่ AI ต้องหาให้เจอ")

### เดินลงเขาด้วย Gradient Descent — ดูเป็นภาพ!

In [ ]:
# --- Gradient Descent เดินลงเขา: plot ลง Contour ---
a = 0.0
b = 0.0
learning_rate = 0.02
epochs = 50
n = len(X_data)

path_a = [a]
path_b = [b]

for _ in range(epochs):
    y_pred = a * X_data + b
    # ใช้สูตรจาก Week 3
    grad_a = -2/n * np.sum(X_data * (Y_data - y_pred))
    grad_b = -2/n * np.sum(Y_data - y_pred)
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    path_a.append(a)
    path_b.append(b)

# --- Plot เส้นทางเดินบน contour ---
fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contour(A, B, Loss, levels=20, cmap='viridis')
ax.contourf(A, B, Loss, levels=20, cmap='viridis', alpha=0.5)

# เส้นทางเดิน
ax.plot(path_a, path_b, 'r.-', markersize=6, linewidth=1.5,
        label=f'เส้นทาง Gradient Descent ({epochs} step)')
ax.scatter([path_a[0]], [path_b[0]], color='orange', s=200,
           marker='o', zorder=5, label='จุดเริ่มต้น')
ax.scatter([path_a[-1]], [path_b[-1]], color='red', s=200,
           marker='*', zorder=5, label='จุดสุดท้าย')

ax.set_xlabel('a (ความชัน)')
ax.set_ylabel('b (จุดตัดแกน y)')
ax.set_title('AI เดินลงหุบเขาด้วย Gradient Descent', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.colorbar(contour, ax=ax)
plt.tight_layout()
plt.show()

print(f"เริ่มต้นที่ a={path_a[0]:.2f}, b={path_b[0]:.2f}")
print(f"จบที่     a={path_a[-1]:.3f}, b={path_b[-1]:.3f}")
print("ความจริง  a=2.00, b=1.00")
print()
print("AI เรียนรู้จริง ๆ ก็คือเดินลงเขาแบบนี้นี่เอง!")

### 1.5 Matrix Operations — ทำนายเยอะๆ ใน 1 บรรทัด

จนถึงตอนนี้ เรา loop ทำนายทีละจุด:
```python
for i in range(len(X)):
    y_pred[i] = a * X[i] + b
```

แต่ในความเป็นจริง ML ใช้ **Matrix** ทำนายหลายจุดพร้อมกันใน 1 บรรทัด — เร็วกว่ามาก

In [ ]:
# --- Matrix Multiplication: ทำนายหลายจุดพร้อมกัน ---

# สมมติมี 5 วัน, 2 feature (อุณหภูมิ, จำนวนลูกค้า)
X = np.array([
    [32, 100],   # วันที่ 1
    [35, 120],
    [28,  75],
    [30,  85],
    [33, 110]
])

# น้ำหนัก (weights) ของแต่ละ feature
W = np.array([0.5, 0.3])
b = 5.0

# --- วิธีเก่า: loop ---
y_pred_loop = []
for row in X:
    y_pred_loop.append(row[0]*W[0] + row[1]*W[1] + b)

# --- วิธีใหม่: 1 บรรทัด! ---
y_pred_matrix = X @ W + b   # @ คือ matrix multiplication

print("X (ข้อมูล):")
print(X)
print()
print(f"ทำนายด้วย loop:   {y_pred_loop}")
print(f"ทำนายด้วย matrix: {y_pred_matrix}")
print()
print("ผลเหมือนกัน! แต่ matrix เขียนสั้นกว่ามาก และเร็วกว่าในข้อมูลจริง")

#### Preview: นี่คือ 1 Layer ของ Neural Network

```python
y = X @ W + b
```

สูตรนี้ก็คือ **1 layer ของ Neural Network** นั่นเอง! ใน Week 9 เราจะต่อยอดเป็น Deep Learning

**สรุปช่วงที่ 1:**

| เรียนไป | หมายความว่า |
|---------|-------------|
| Derivative = ความชัน | Gradient คือความชันของ Loss |
| Loss Landscape 3D | AI เดินลงหุบเขาหาจุดต่ำสุด |
| SymPy | Python หา derivative ให้ได้ ไม่ต้องคำนวณเอง |
| Matrix `X @ W + b` | ทำนายเยอะๆ ใน 1 บรรทัด = หัวใจ Neural Network |

---
### Workshop 1: หาจุดต่ำสุดของฟังก์ชันด้วย Gradient Descent + SymPy
---

**โจทย์:** ให้ `f(x) = x² - 4x + 5`

1. ใช้ SymPy หา derivative ของ f(x)
2. ใช้ Gradient Descent หาจุดต่ำสุด โดยเริ่มที่ x = 0
3. ใช้ learning_rate = 0.1, epochs = 20

In [ ]:
# ===========================================
# Workshop 1: ลองเขียนเองตรงนี้
# ===========================================

import sympy as sp

# 1. หา derivative ด้วย SymPy
x_sym = sp.Symbol('x')

# TODO: เขียนฟังก์ชัน f(x) = x² - 4x + 5
# f = ???

# TODO: หา derivative
# f_prime = sp.diff(???, ???)

# print(f"f(x) = {f}")
# print(f"f'(x) = {f_prime}")


# 2. เปลี่ยน derivative เป็น function ที่ใช้งานกับตัวเลขได้
# gradient_func = sp.lambdify(x_sym, f_prime)


# 3. Gradient Descent
# x = 0.0
# lr = ???
# epochs = ???

# for epoch in range(epochs):
#     grad = gradient_func(x)
#     x = x - lr * grad
#     print(f"Epoch {epoch+1}: x = {x:.4f}")

# print(f"\nจุดต่ำสุด: x = {x:.4f}")


In [ ]:
# ===========================================
# Workshop 1: เฉลย
# ===========================================

import sympy as sp

# 1. หา derivative ด้วย SymPy
x_sym = sp.Symbol('x')
f = x_sym**2 - 4*x_sym + 5
f_prime = sp.diff(f, x_sym)

print(f"f(x) = {f}")
print(f"f'(x) = {f_prime}")
print()

# 2. เปลี่ยน derivative เป็น function ที่ใช้งานกับตัวเลขได้
gradient_func = sp.lambdify(x_sym, f_prime)

# 3. Gradient Descent
x = 0.0
lr = 0.1
epochs = 20

print(f"เริ่มต้น: x = {x}")
print("-" * 40)

for epoch in range(epochs):
    grad = gradient_func(x)
    x = x - lr * grad
    if epoch % 3 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1:2d}: x = {x:.4f}, grad = {grad:.4f}")

print("-" * 40)
print(f"\nจุดต่ำสุด: x = {x:.4f}")
print(f"f(x) ที่จุดต่ำสุด = {x**2 - 4*x + 5:.4f}")
print(f"\nเฉลยจริง: จุดต่ำสุดคือ x = 2, f(2) = 1")

---
## ช่วงที่ 2: เปิดโลก Data จริง -> ค่าน้ำมัน
---

**สถานการณ์:**

> เราเป็น Data Scientist ของบริษัทขนส่ง
> หัวหน้าให้ข้อมูลการเติมน้ำมันของรถ 500 รายการมา
> "ช่วยเอาไปทำ AI ทำนายค่าน้ำมันของเดือนหน้าหน่อยนะ"

เราเปิดไฟล์มาดู... **"ข้อมูลเละเทะมาก!"** — ที่เห็นในหนังสือเรียน มีแต่ข้อมูลสะอาดแล้วทั้งนั้น แต่โลกจริงไม่เป็นแบบนั้น

ข้อมูลจริง มักมีปัญหาพวกนี้:
- ข้อมูลขาด (Missing)
- ข้อมูลซ้ำ (Duplicate)
- ค่าผิดปกติ (Outlier)
- ประเภทข้อมูลผิด (Type mismatch) เช่น ตัวเลขเขียนเป็น string "43.45 บาท"
- เขียนไม่เหมือนกัน เช่น "PTT", "ปตท.", "Ptt"

**Week 4 ทั้งหมดคือการแก้ปัญหาพวกนี้!**

### 2.1 สร้าง Dataset ค่าน้ำมัน (แบบสกปรก)

เราจำลองข้อมูลให้ใกล้เคียงของจริง — มีทั้งความสกปรกและ pattern ซ่อนอยู่

In [ ]:
# --- สร้าง Dataset ค่าน้ำมัน (แบบสกปรก เหมือนของจริง) ---
np.random.seed(42)
n = 500

# ช่วงราคาน้ำมันไทย (อ้างอิง เม.ย. 2569)
# ดีเซล ~44, แก๊สโซฮอล์ 95 ~43, เบนซิน ~50
fuel_types = ['ดีเซล', 'แก๊สโซฮอล์ 95', 'แก๊สโซฮอล์ 91', 'เบนซิน']
fuel_prices = {'ดีเซล': 44.0, 'แก๊สโซฮอล์ 95': 43.5, 'แก๊สโซฮอล์ 91': 42.8, 'เบนซิน': 50.2}

# ปั๊ม (เขียนไม่เหมือนกันเพื่อจำลองข้อมูลจริง)
stations = ['PTT', 'ปตท.', 'Bangchak', 'บางจาก', 'Shell', 'เชลล์', 'Caltex']

# สร้าง base data
dates = pd.date_range('2025-10-01', periods=n, freq='D')
dates = np.random.choice(dates, n)

data = {
    'date': dates,
    'station': np.random.choice(stations, n),
    'fuel_type': np.random.choice(fuel_types, n, p=[0.45, 0.30, 0.15, 0.10]),
    'car_type': np.random.choice(['รถเก๋ง', 'รถกระบะ', 'รถตู้', 'รถบรรทุก'], n, p=[0.4, 0.35, 0.15, 0.10]),
    'province': np.random.choice(['กรุงเทพ', 'นนทบุรี', 'ปทุมธานี', 'สมุทรปราการ', 'ชลบุรี'], n),
}

df = pd.DataFrame(data)

# สร้างคอลัมน์ที่สัมพันธ์กัน
df['liters'] = np.round(np.random.uniform(20, 60, n), 2)
df['price_per_liter'] = df['fuel_type'].map(fuel_prices) + np.random.normal(0, 0.5, n)
df['price_per_liter'] = df['price_per_liter'].round(2)

# ระยะทางวิ่ง (สัมพันธ์กับประเภทรถ)
base_km = {'รถเก๋ง': 400, 'รถกระบะ': 500, 'รถตู้': 450, 'รถบรรทุก': 600}
df['km_driven'] = df['car_type'].map(base_km) + np.random.normal(0, 80, n)
df['km_driven'] = df['km_driven'].round(0).astype(int)

# Total cost = liters * price_per_liter (+ noise)
df['total_cost'] = (df['liters'] * df['price_per_liter'] + np.random.normal(0, 10, n)).round(2)

# ===== ใส่ความสกปรกเข้าไป (จำลองข้อมูลจริง) =====
# 1. Missing values
missing_idx_1 = np.random.choice(n, 30, replace=False)
df.loc[missing_idx_1, 'liters'] = np.nan
missing_idx_2 = np.random.choice(n, 20, replace=False)
df.loc[missing_idx_2, 'km_driven'] = np.nan

# 2. Duplicate rows
dup_idx = np.random.choice(n, 15, replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

# 3. Outliers (เติม 200 ลิตรใน 1 ครั้ง = ผิดปกติ)
outlier_idx = np.random.choice(len(df), 5, replace=False)
df.loc[outlier_idx, 'liters'] = np.random.uniform(150, 250, 5)
df.loc[outlier_idx, 'total_cost'] = df.loc[outlier_idx, 'liters'] * df.loc[outlier_idx, 'price_per_liter']

# 4. Type mismatch: ทำ price_per_liter บางค่าเป็น string "XX.XX บาท"
# เปลี่ยน column type เป็น object ก่อน เพื่อให้รวม string กับ float ได้
df['price_per_liter'] = df['price_per_liter'].astype(object)
str_idx = np.random.choice(len(df), 25, replace=False)
for idx in str_idx:
    df.at[idx, 'price_per_liter'] = f"{df.at[idx, 'price_per_liter']:.2f} บาท"

# สลับลำดับ
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"โหลดข้อมูลเสร็จ! มีทั้งหมด {len(df)} รายการ")
print("\nตัวอย่างข้อมูล 5 แถวแรก:")
df.head()

### 2.2 ดูข้อมูลดิบ — เจออะไรบ้าง?

In [ ]:
# --- สำรวจข้อมูลเบื้องต้น ---
print("=" * 50)
print("SHAPE:", df.shape)
print("=" * 50)

print("\n--- ข้อมูลแต่ละคอลัมน์ ---")
df.info()

In [ ]:
# --- สรุปสถิติ ---
print("\n--- สถิติ (เฉพาะคอลัมน์ตัวเลข) ---")
df.describe()

In [ ]:
# --- ตรวจ Missing Data ---
print("จำนวน Missing Data แต่ละคอลัมน์:")
print(df.isnull().sum())

print("\n--- ตรวจ Duplicate ---")
print(f"จำนวนแถวที่ซ้ำ: {df.duplicated().sum()}")

**สังเกต 4 ปัญหา:**

1.  คอลัมน์ `liters` มี NaN อยู่ ~30 แถว
2.  คอลัมน์ `km_driven` มี NaN อยู่ ~20 แถว
3.  มีแถวซ้ำ ~15 แถว
4.  คอลัมน์ `price_per_liter` เป็น `object` (string) ไม่ใช่ตัวเลข — เพราะบางแถวเขียน "43.45 บาท"

Week 4 ที่เหลือ เราจะแก้ปัญหาพวกนี้ทีละอย่าง

---
## ช่วงที่ 3: Data Cleaning — ทำความสะอาดข้อมูล
---

เป็นขั้นตอนที่ **ใช้เวลามากที่สุดในงาน Data Science จริง** (~70% ของเวลาทั้งหมด)

ถ้าข้อมูลไม่สะอาด ต่อให้โมเดลดีแค่ไหน ผลก็แย่ — **"Garbage in, Garbage out"**

### 3.1 แก้ประเภทข้อมูลผิด (Type Mismatch)

ปัญหา: `price_per_liter` ควรเป็นตัวเลข แต่บางแถวเขียน "43.45 บาท" → Pandas อ่านเป็น string

In [ ]:
# --- ดูตัวอย่างค่าที่มีปัญหา ---
print("ตัวอย่างค่าใน price_per_liter:")
print(df['price_per_liter'].head(20).tolist())
print(f"\nType: {df['price_per_liter'].dtype}")

In [ ]:
# --- แก้: ลบคำว่า "บาท" ออก แล้วแปลงเป็นตัวเลข ---
df['price_per_liter'] = (df['price_per_liter']
.astype(str) # แน่ใจว่าเป็น string ก่อน
.str.replace(' บาท', '') # ลบ " บาท" ออก
.str.replace('บาท', '') # กันไว้กรณีไม่มีเว้นวรรค
.astype(float)) # แปลงเป็นตัวเลข

print(f"Type หลังแก้: {df['price_per_liter'].dtype}")
print(f"ตัวอย่าง: {df['price_per_liter'].head().tolist()}")
print("\nแก้เสร็จ! ตอนนี้เป็นตัวเลขแล้ว")

### 3.2 ลบแถวที่ซ้ำ (Duplicates)

In [ ]:
# --- ตรวจและลบแถวซ้ำ ---
before = len(df)
print(f"ก่อน: {before} แถว")
print(f"แถวซ้ำ: {df.duplicated().sum()} แถว")

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)
print(f"หลัง: {after} แถว (ลบไป {before - after} แถว)")
print("\nลบแถวซ้ำเสร็จ!")

### 3.3 จัดการ Missing Data

**3 วิธีหลัก:**

| วิธี | เมื่อไหร่ใช้ |
|------|--------------|
| `dropna()` | Missing น้อยมาก (<5%) หรือแถวนั้นสำคัญน้อย |
| `fillna(mean)` | Numeric + distribution ปกติ |
| `fillna(median)` | Numeric + มี outlier |
| `fillna(mode)` | Categorical |

สำหรับ **ค่าน้ำมัน** — เราใช้ **median** เพราะมี outlier

In [ ]:
# --- ดู missing values ก่อนแก้ ---
print("Missing data ตอนนี้:")
print(df.isnull().sum())
print(f"\nรวม {df.isnull().any(axis=1).sum()} แถว ที่มี missing")

In [ ]:
# --- เติม missing ด้วย median ---
# liters -> median
liters_median = df['liters'].median()
df['liters'] = df['liters'].fillna(liters_median)
print(f"เติม liters ด้วย median = {liters_median:.2f}")

# km_driven -> median แยกตามประเภทรถ (ดีกว่าใช้ median รวม)
df['km_driven'] = df.groupby('car_type')['km_driven'].transform(
lambda x: x.fillna(x.median())
)
print("เติม km_driven ด้วย median แยกตาม car_type")

print("\nMissing หลังแก้:")
print(df.isnull().sum())
print("\nไม่มี missing แล้ว!")

**เทคนิค pro:** `groupby().transform()` — เติม median แยกตามกลุ่ม
เช่น รถกระบะใช้ median ของรถกระบะ, รถบรรทุกใช้ median ของรถบรรทุก — แม่นกว่าใช้ median รวม

### 3.4 ตรวจและจัดการ Outliers

**Outlier** = ค่าผิดปกติ (อาจเกิดจากกรอกผิด, เครื่องวัดเสีย, หรือเคสพิเศษ)

**วิธีตรวจ: IQR (Interquartile Range)**
- Q1 = ค่าที่ 25%
- Q3 = ค่าที่ 75%
- IQR = Q3 - Q1
- Outlier = ค่าที่ < Q1 - 1.5×IQR หรือ > Q3 + 1.5×IQR

In [ ]:
# --- ดู liters ที่น่าจะมี outlier ---
print(f"Min:    {df['liters'].min():.2f}")
print(f"Max:    {df['liters'].max():.2f}")
print(f"Mean:   {df['liters'].mean():.2f}")
print(f"Median: {df['liters'].median():.2f}")

# Plot box plot เพื่อดู outlier
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(df['liters'], vert=False)
axes[0].set_title('Box Plot: liters (ดู outlier)', fontsize=12)
axes[0].set_xlabel('จำนวนลิตร')
axes[0].grid(True, alpha=0.3)

axes[1].hist(df['liters'], bins=50, color='#3498db', edgecolor='white')
axes[1].set_title('Histogram: liters', fontsize=12)
axes[1].set_xlabel('จำนวนลิตร')
axes[1].set_ylabel('ความถี่')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nเห็นไหม? มี outlier ที่เติม 200+ ลิตร ซึ่งผิดปกติมาก (ถังรถทั่วไปแค่ 50-70 ลิตร)")

In [ ]:
# --- ใช้ IQR หา outlier ---
Q1 = df['liters'].quantile(0.25)
Q3 = df['liters'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:.2f}")
print(f"Q3 = {Q3:.2f}")
print(f"IQR = {IQR:.2f}")
print(f"ช่วงปกติ: {lower_bound:.2f} - {upper_bound:.2f}")

# หา outlier
outliers = df[(df['liters'] < lower_bound) | (df['liters'] > upper_bound)]
print(f"\nพบ Outliers: {len(outliers)} แถว")
print(outliers[['date', 'fuel_type', 'liters', 'total_cost']].head())

In [ ]:
# --- ตัดสินใจ: ลบ outliers (เพราะน่าจะเป็นข้อมูลผิดจริง ๆ) ---
before = len(df)
df = df[(df['liters'] >= lower_bound) & (df['liters'] <= upper_bound)].reset_index(drop=True)
after = len(df)

print(f"ลบ outliers ไป {before - after} แถว")
print(f"ข้อมูลเหลือ: {after} แถว")
print(f"\nMax liters ใหม่: {df['liters'].max():.2f} (ปกติแล้ว)")

### 3.5 ทำข้อความให้เหมือนกัน (Text Normalization)

ปัญหา: "PTT", "ปตท.", "Ptt" — เป็นปั๊มเดียวกันแต่เขียนคนละแบบ

In [ ]:
# --- ดูค่าที่ไม่ซ้ำในคอลัมน์ station ---
print("ปั๊มที่มีในข้อมูล:")
print(df['station'].value_counts())

In [ ]:
# --- Map ให้เป็นชื่อเดียวกัน ---
station_map = {
    'PTT': 'ปตท.',
    'ปตท.': 'ปตท.',
    'Bangchak': 'บางจาก',
    'บางจาค': 'บางจาก',
    'Shell': 'เชลล์',
    'เชลล์': 'เชลล์',
    'Caltex': 'คาลเท็กซ์',
}

df['station'] = df['station'].map(station_map)

print("หลังทำความสะอาด:")
print(df['station'].value_counts())
print("\nตอนนี้ชื่อปั๊มเหมือนกันแล้ว")

---
### Workshop 2: ทำความสะอาดข้อมูลน้ำมันเวอร์ชันใหม่
---

**โจทย์:** สร้าง DataFrame ใหม่ที่มีปัญหา แล้วลองทำความสะอาด

```python
messy = pd.DataFrame({
    'price': ['45.20 บาท', '43.00 บาท', np.nan, '46.50 บาท', '44.00 บาท', '45.20 บาท'],
    'liters': [40, 50, 45, np.nan, 300, 40],  # 300 คือ outlier
    'type':   ['ดีเซล', 'Diesel', 'ดีเซล', 'ดีเซล', 'ดีเซล', 'ดีเซล'],  # "Diesel" ต้องแมปเป็น "ดีเซล"
})
```

1. แก้ `price` ให้เป็นตัวเลข (ลบ "บาท")
2. เติม missing ของ `liters` ด้วย median
3. ตัด outlier ของ `liters` (ใช้ IQR หรือกำหนดเองว่า liters <= 100)
4. ทำ `type` ให้เหมือนกัน (Diesel → ดีเซล)
5. ลบแถวซ้ำ

In [ ]:
# ===========================================
# Workshop 2: ลองเขียนเองตรงนี้
# ===========================================

messy = pd.DataFrame({
    'price': ['45.20 บาท', '43.00 บาท', np.nan, '46.50 บาท', '44.00 บาท', '45.20 บาท'],
    'liters': [40, 50, 45, np.nan, 300, 40],
    'type':   ['ดีเซล', 'Diesel', 'ดีเซล', 'ดีเซล', 'ดีเซล', 'ดีเซล'],
})

print("ข้อมูลก่อนแก้:")
print(messy)

# TODO: 1. แก้ price ให้เป็นตัวเลข

# TODO: 2. เติม missing ของ liters ด้วย median

# TODO: 3. ตัด outlier ของ liters (liters <= 100)

# TODO: 4. ทำ type ให้เหมือนกัน

# TODO: 5. ลบแถวซ้ำ

# print("\nข้อมูลหลังแก้:")
# print(messy)

In [ ]:
# ===========================================
# Workshop 2: เฉลย
# ===========================================

messy = pd.DataFrame({
    'price': ['45.20 บาท', '43.00 บาท', np.nan, '46.50 บาท', '44.00 บาท', '45.20 บาท'],
    'liters': [40, 50, 45, np.nan, 300, 40],
    'type':   ['ดีเซล', 'Diesel', 'ดีเซล', 'ดีเซล', 'ดีเซล', 'ดีเซล'],
})

print("ข้อมูลก่อนแก้:")
print(messy)

# 1. แก้ price (ต้อง fill NaN ก่อน ไม่งั้น str accessor ทำงานไม่ได้)
messy['price'] = messy['price'].fillna('0 บาท')
messy['price'] = messy['price'].str.replace(' บาท', '').astype(float)
messy.loc[messy['price'] == 0, 'price'] = np.nan  # เปลี่ยน 0 กลับเป็น NaN
messy['price'] = messy['price'].fillna(messy['price'].median())

# 2. เติม missing ของ liters
messy['liters'] = messy['liters'].fillna(messy['liters'].median())

# 3. ตัด outlier
messy = messy[messy['liters'] <= 100].reset_index(drop=True)

# 4. ทำ type ให้เหมือนกัน
messy['type'] = messy['type'].replace({'Diesel': 'ดีเซล'})

# 5. ลบแถวซ้ำ
messy = messy.drop_duplicates().reset_index(drop=True)

print("\nข้อมูลหลังแก้:")
print(messy)
print(f"\nเหลือ {len(messy)} แถวที่สะอาด")

---
## ช่วงที่ 4: Feature Engineering — สร้าง Feature ใหม่ให้ AI ฉลาดขึ้น
---

**Feature Engineering** = การสร้าง/แปลง feature ให้ AI เข้าใจข้อมูลง่ายขึ้น

ตัวอย่างจริง:
- **Netflix** ไม่ได้แค่ใช้ "ชื่อหนัง" — สร้าง feature อย่าง "จำนวน view ใน 7 วันล่าสุด", "คะแนนเฉลี่ยของ user คนเดียวกัน"
- **Grab** ไม่ได้แค่ใช้ "ระยะทาง" — สร้าง feature อย่าง "เวลาวันในสัปดาห์", "ฝนตกมั้ย"

**Feature ที่ดี > อัลกอริทึมที่ซับซ้อน** — นี่คือหัวใจของ ML ในโลกจริง

### 4.1 สร้าง Feature จากวันที่

วันที่ `2025-10-15` มีข้อมูลซ่อนอยู่เยอะมาก:
- วันในสัปดาห์ (จันทร์-อาทิตย์)
- เดือน
- วันหยุดหรือไม่
- ต้นเดือน/ปลายเดือน

In [ ]:
# --- แปลงเป็น datetime ก่อน ---
df['date'] = pd.to_datetime(df['date'])

# สร้าง feature ใหม่จากวันที่
df['day_of_week'] = df['date'].dt.day_name()        # จันทร์, อังคาร, ...
df['month'] = df['date'].dt.month                    # 1-12
df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)  # เสาร์-อาทิตย์

print("ข้อมูลหลังสร้าง feature ใหม่:")
print(df[['date', 'day_of_week', 'month', 'is_weekend']].head(10))

### 4.2 สร้าง Feature จากการคำนวณ

Feature ใหม่จากข้อมูลเดิม:
- **อัตราสิ้นเปลือง** = ระยะทาง / ลิตร (km/L)
- **ค่าน้ำมันต่อ 100 กม.** = (total_cost / km_driven) × 100

In [ ]:
# --- สร้าง feature ที่คำนวณได้ ---
df['km_per_liter'] = (df['km_driven'] / df['liters']).round(2)
df['cost_per_100km'] = (df['total_cost'] / df['km_driven'] * 100).round(2)

print("Feature ที่สร้างใหม่:")
print(df[['car_type', 'km_driven', 'liters', 'km_per_liter', 'cost_per_100km']].head(10))

print("\nอัตราสิ้นเปลืองเฉลี่ยของแต่ละประเภทรถ:")
print(df.groupby('car_type')['km_per_liter'].mean().round(2))

### 4.3 One-Hot Encoding — แปลง Text เป็นตัวเลข

AI เข้าใจแต่ตัวเลข — ต้องแปลง category (เช่น "ดีเซล", "เบนซิน") เป็นตัวเลขก่อน

**วิธีที่ผิด:** เขียน "ดีเซล"=1, "เบนซิน"=2, "แก๊สโซฮอล์"=3  
**ทำไมผิด?** AI จะคิดว่า เบนซิน > ดีเซล, แก๊สโซฮอล์ > เบนซิน — ทั้งที่มันไม่มีลำดับกัน

**วิธีถูก: One-Hot Encoding** — สร้างคอลัมน์ใหม่ 0/1 สำหรับแต่ละหมวด

| ต้นฉบับ | ดีเซล | เบนซิน | แก๊สโซฮอล์ |
|---------|:-----:|:------:|:---------:|
| ดีเซล   | 1 | 0 | 0 |
| เบนซิน  | 0 | 1 | 0 |

In [ ]:
# --- One-Hot Encoding ด้วย pd.get_dummies ---
fuel_dummies = pd.get_dummies(df['fuel_type'], prefix='fuel').astype(int)
print("คอลัมน์ใหม่ที่สร้าง:")
print(fuel_dummies.head())

In [ ]:
# --- ทำกับหลายคอลัมน์พร้อมกัน ---
df_encoded = pd.get_dummies(
    df,
    columns=['fuel_type', 'car_type', 'station', 'province'],
    drop_first=False
).astype({col: int for col in df.select_dtypes('bool').columns} if False else {})

# แปลง bool เป็น int
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'bool':
        df_encoded[col] = df_encoded[col].astype(int)

print(f"คอลัมน์เดิม: {df.shape[1]} คอลัมน์")
print(f"คอลัมน์ใหม่: {df_encoded.shape[1]} คอลัมน์")
print("\nคอลัมน์ที่ถูก encode:")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
for c in new_cols[:10]:
    print(f"  - {c}")
print(f"  ... (รวม {len(new_cols)} คอลัมน์)")

### 4.4 Binning — แบ่งค่าต่อเนื่องเป็นช่วง

บางครั้งตัวเลขต่อเนื่อง (เช่น ราคา, อายุ) ทำงานดีกว่าเมื่อ "แบ่งช่วง"

ตัวอย่าง: liters → `[น้อย, ปานกลาง, มาก, เต็มถัง]`

In [ ]:
# --- Binning: แบ่งจำนวนลิตรเป็น 4 ช่วง ---
df['liter_level'] = pd.cut(
    df['liters'],
    bins=[0, 25, 40, 55, 100],
    labels=['น้อย', 'ปานกลาง', 'มาก', 'เต็มถัง']
)

print(df[['liters', 'liter_level']].head(10))
print("\nจำนวนแต่ละช่วง:")
print(df['liter_level'].value_counts())

### 4.5 Feature Scaling — เชื่อมกลับมาที่ Math ช่วงแรก!

**ปัญหา:** feature มี scale ต่างกันมาก
- `km_driven`: 300 - 700 km
- `liters`: 20 - 60 L
- `is_weekend`: 0 หรือ 1

**ผลเสีย:** Gradient Descent จะ "zigzag" — ช้า บางทีไม่ converge เลย

**วิธีแก้:** Scale ให้อยู่ในช่วงใกล้กัน

**2 วิธีมาตรฐาน:**

| วิธี | สูตร | ช่วงผลลัพธ์ | เหมาะกับ |
|------|-----|-----------|----------|
| **Min-Max Normalization** | `(x - min) / (max - min)` | [0, 1] | Neural Network, ไม่มี outlier |
| **Standardization (Z-score)** | `(x - mean) / std` | mean=0, std=1 | Linear, Logistic, SVM |

**ด้านล่างเราจะ plot Loss Landscape + เส้นทาง Gradient Descent ก่อน/หลัง Scale เพื่อให้เห็นผลจริง**

In [ ]:
# --- Demo: ทำไม Scaling ถึงช่วย Gradient Descent ---
# เราจะเปรียบเทียบ Loss Landscape ก่อน/หลัง Scaling
# และดูว่า Gradient Descent เดินยังไง

from sklearn.preprocessing import StandardScaler

np.random.seed(0)
n = 80

# สร้างข้อมูล 2 feature ที่ scale ต่างกัน 10 เท่า
# x1 อยู่ช่วง [-1, 1]   (เช่น is_weekend)
# x2 อยู่ช่วง [-10, 10] (เช่น liters - mean)
x1 = np.random.uniform(-1, 1, n)
x2 = np.random.uniform(-10, 10, n)

# สร้าง y ตามสมการจริง
y_true_w1, y_true_w2 = 2.0, 0.5
y = y_true_w1 * x1 + y_true_w2 * x2 + np.random.normal(0, 0.3, n)

# Scale ข้อมูลด้วย StandardScaler
scaler = StandardScaler()
X_raw = np.column_stack([x1, x2])
X_scaled = scaler.fit_transform(X_raw)
x1_s, x2_s = X_scaled[:, 0], X_scaled[:, 1]

# ฟังก์ชันคำนวณ Loss สำหรับสร้าง contour
def compute_loss_grid(X1, X2, y, w1_range, w2_range):
    W1, W2 = np.meshgrid(w1_range, w2_range)
    L = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            pred = W1[i, j] * X1 + W2[i, j] * X2
            L[i, j] = np.mean((y - pred) ** 2)
    return W1, W2, L

# Loss landscape ก่อน scale (จะเป็นวงรียาว)
W1a, W2a, La = compute_loss_grid(x1, x2, y,
                                  np.linspace(-4, 6, 80),
                                  np.linspace(-0.5, 1.5, 80))

# Loss landscape หลัง scale (จะเป็นวงกลม)
W1b, W2b, Lb = compute_loss_grid(x1_s, x2_s, y,
                                  np.linspace(-2, 6, 60),
                                  np.linspace(-2, 6, 60))

# ฟังก์ชัน Gradient Descent
def run_gd(X1, X2, y, w1_start, w2_start, lr, epochs):
    w1, w2 = w1_start, w2_start
    hist_w1, hist_w2 = [w1], [w2]
    n = len(y)
    for _ in range(epochs):
        pred = w1 * X1 + w2 * X2
        grad_w1 = -2/n * np.sum(X1 * (y - pred))
        grad_w2 = -2/n * np.sum(X2 * (y - pred))
        w1 -= lr * grad_w1
        w2 -= lr * grad_w2
        hist_w1.append(w1)
        hist_w2.append(w2)
    return hist_w1, hist_w2

# รัน Gradient Descent ก่อน scale
# ใช้ lr ที่พอดี (ใหญ่กว่านี้ diverge, เล็กกว่านี้ไม่ zigzag)
h1a, h2a = run_gd(x1, x2, y, w1_start=5.5, w2_start=1.2, lr=0.025, epochs=60)

# รัน Gradient Descent หลัง scale
# ใช้ lr ปกติได้ เพราะ feature scale เท่ากันแล้ว
h1b, h2b = run_gd(x1_s, x2_s, y, w1_start=5.5, w2_start=5.5, lr=0.05, epochs=30)

# Plot เปรียบเทียบ
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- ก่อน scale ---
axes[0].contour(W1a, W2a, La, levels=15, cmap='viridis')
axes[0].contourf(W1a, W2a, La, levels=15, cmap='viridis', alpha=0.4)
axes[0].plot(h1a, h2a, 'r.-', markersize=5, linewidth=1.5, label='Gradient Descent')
axes[0].scatter([h1a[0]], [h2a[0]], color='orange', s=150, marker='o',
                zorder=5, label='จุดเริ่มต้น')
axes[0].scatter([h1a[-1]], [h2a[-1]], color='red', s=200, marker='*',
                zorder=5, label='จุดสุดท้าย')
axes[0].set_title('ก่อน Scaling: Loss Landscape เป็นวงรียาว\n'
                  'Gradient Descent เดินแบบ zigzag ช้ามาก',
                  fontsize=12)
axes[0].set_xlabel('w1 (น้ำหนักของ feature 1)')
axes[0].set_ylabel('w2 (น้ำหนักของ feature 2)')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# --- หลัง scale ---
axes[1].contour(W1b, W2b, Lb, levels=15, cmap='viridis')
axes[1].contourf(W1b, W2b, Lb, levels=15, cmap='viridis', alpha=0.4)
axes[1].plot(h1b, h2b, 'r.-', markersize=5, linewidth=1.5, label='Gradient Descent')
axes[1].scatter([h1b[0]], [h2b[0]], color='orange', s=150, marker='o',
                zorder=5, label='จุดเริ่มต้น')
axes[1].scatter([h1b[-1]], [h2b[-1]], color='red', s=200, marker='*',
                zorder=5, label='จุดสุดท้าย')
axes[1].set_title('หลัง Scaling: Loss Landscape เป็นวงกลม\n'
                  'Gradient Descent เดินตรงลงก้นหุบเขา',
                  fontsize=12)
axes[1].set_xlabel('w1 (scaled)')
axes[1].set_ylabel('w2 (scaled)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

loss_before = np.mean((y - h1a[-1]*x1 - h2a[-1]*x2)**2)
loss_after = np.mean((y - h1b[-1]*x1_s - h2b[-1]*x2_s)**2)

print("เปรียบเทียบผลลัพธ์:")
print(f"  ก่อน scale: ใช้ {len(h1a)-1} step, Loss สุดท้าย = {loss_before:.3f}")
print(f"             เส้นทางคดเคี้ยว (zigzag) เพราะหุบเขายืดยาว")
print(f"  หลัง scale: ใช้ {len(h1b)-1} step, Loss สุดท้าย = {loss_after:.3f}")
print(f"             เส้นทางตรงสวย เพราะหุบเขาเป็นวงกลม")
print()
print(f"สรุป: หลัง scale ใช้ step น้อยกว่า {(len(h1a)-1)//(len(h1b)-1)} เท่า และได้ Loss ต่ำกว่า!")

In [ ]:
# --- ทำจริงกับข้อมูลน้ำมัน ---
from sklearn.preprocessing import StandardScaler

# เลือก feature ที่เป็นตัวเลขต่อเนื่อง
numeric_features = ['liters', 'price_per_liter', 'km_driven', 'km_per_liter', 'cost_per_100km']

print("ก่อน scaling:")
print(df[numeric_features].describe().round(2))

# Scale
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_features] = scaler.fit_transform(df[numeric_features])

print("\nหลัง scaling (mean~=0, std~=1):")
print(df_scaled[numeric_features].describe().round(2))

---
## ช่วงที่ 5: กับดัก 3 ตัวที่ทำลาย ML Model
---

รู้จัก Clean + Feature Engineering แล้ว — แต่ยังมีกับดักอีก 3 ตัวที่มือใหม่ติดเยอะมาก

1. **Bias** — ข้อมูลเอียง
2. **Noise** — ข้อมูลสกปรก
3. **Data Leakage** — ข้อมูลรั่ว (อันตรายที่สุด!)

### 5.1 Bias — ข้อมูลเอียง

**ตัวอย่าง:** เก็บข้อมูลค่าน้ำมันเฉพาะรถเก๋งในกรุงเทพ → ใช้ทำนายรถบรรทุกทั่วประเทศ

**ผลลัพธ์:** โมเดลทำนายผิดมาก เพราะ:
- รถเก๋ง ≠ รถบรรทุก (ขนาดเครื่องต่างกัน)
- กรุงเทพ ≠ ต่างจังหวัด (ราคาน้ำมันต่างกันเล็กน้อย, รถติดต่างกัน)

**วิธีป้องกัน:**
- เก็บข้อมูลให้ครอบคลุม (ทุกประเภทรถ, ทุกจังหวัด)
- ตรวจสัดส่วนของแต่ละกลุ่มในข้อมูล

In [ ]:
# --- ตรวจ bias ในข้อมูล ---
print("สัดส่วนประเภทรถ:")
print(df['car_type'].value_counts(normalize=True).round(2))

print("\nสัดส่วนจังหวัด:")
print(df['province'].value_counts(normalize=True).round(2))

print("\nสัดส่วนชนิดน้ำมัน:")
print(df['fuel_type'].value_counts(normalize=True).round(2))

**ตีความ:** ถ้าพบว่าบางกลุ่มมีแค่ 5% — อาจต้องเก็บข้อมูลเพิ่ม หรือระวังว่าโมเดลจะเก่งแค่เฉพาะกลุ่มใหญ่

### 5.2 Noise — ข้อมูลสกปรก

**ตัวอย่าง Noise:**
- Sensor วัดผิด: ควรเติม 40 ลิตร แต่บันทึก 400
- พนักงานกดผิด: ราคา 43.50 → 435
- Internet ตัดระหว่างบันทึก → ข้อมูลไม่ครบ

**วิธีจัดการ:** ทำใน Section 3 แล้ว (Outlier detection, Missing imputation)

> **Bias vs Noise:**  
> - Bias = ข้อมูลไม่ครบ/เอียง (**systematic**)  
> - Noise = ข้อมูลผิดเป็นจุด ๆ (**random**)

### 5.3 Data Leakage — กับดักที่อันตรายที่สุด!

**Data Leakage** = ข้อมูลที่ไม่ควรรู้ตอน train รั่วเข้าไปในการ train

**ผลลัพธ์:** โมเดลทำคะแนนสูงมากตอน test แต่**ใช้งานจริงพัง**!

---

**กับดักคลาสสิค #1: Scale ก่อน Split**

**ผิด:**
```python
scaler.fit(X) # ใช้ข้อมูลทั้งหมด (train+test)
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled) # test รู้ mean/std ของ train แล้ว!
```

**ถูก:**
```python
X_train, X_test = train_test_split(X) # split ก่อน
scaler.fit(X_train) # fit บน train เท่านั้น
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test) # apply บน test
```

In [ ]:
# --- Demo: ผิด vs ถูก ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ข้อมูลจำลอง
np.random.seed(0)
X = np.random.normal(50, 10, (100, 1))

# ------- วิธีผิด: Scale ก่อน Split -------
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)
X_train_w, X_test_w = train_test_split(X_scaled_wrong, test_size=0.2, random_state=0)

print("วิธีผิด (scale ก่อน split):")
print(f"   scaler รู้ mean ของข้อมูลทั้งหมด = {scaler_wrong.mean_[0]:.2f}")
print(f"   -> ข้อมูล test รั่วเข้ามาช่วย scaler แล้ว!")

# ------- วิธีถูก: Split ก่อน Scale -------
X_train, X_test = train_test_split(X, test_size=0.2, random_state=0)

scaler_right = StandardScaler()
scaler_right.fit(X_train)                  # fit บน train เท่านั้น!
X_train_r = scaler_right.transform(X_train)
X_test_r = scaler_right.transform(X_test)  # apply บน test

print("\nวิธีถูก (split ก่อน scale):")
print(f"   scaler รู้ mean เฉพาะของ train = {scaler_right.mean_[0]:.2f}")
print(f"   -> ไม่มี leakage!")

**กับดักคลาสสิค #2: ใส่ feature ที่ "รู้ผลลัพธ์อยู่แล้ว"**

ตัวอย่าง: ทำนาย **ค่าน้ำมันเดือนหน้า**  
แต่ใส่ feature "ยอดบิลเดือนหน้า" (ซึ่งคือสิ่งที่เราจะทำนาย) → โมเดลแม่น 100% แต่ใช้งานจริงไม่ได้เพราะไม่รู้ยอดเดือนหน้าจนกว่าจะจ่าย

**วิธีเช็ค:** ถามตัวเองว่า "feature นี้ตอนทำนายจริง ๆ เราจะรู้หรือยัง?"

---

**กฎเหล็ก Data Leakage:**

> **ข้อมูลที่ test set หรืออนาคต ต้อง "ไม่รู้" ตอน train**

---
## ช่วงที่ 6: Mini Project — สร้าง Data Prep Pipeline ครบวงจร
---

รวมทุกอย่างที่เรียนมาเป็น **pipeline เดียว** — พร้อมส่งต่อให้ ML Week 5!

```
Raw Data -> Clean -> Feature Engineering -> Split -> Scale -> พร้อม Train!
```

In [ ]:
# ============================================
# FULL DATA PREP PIPELINE - ค่าน้ำมัน
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ========== STEP 1: Load Raw Data ==========
# (ในชีวิตจริงคือ pd.read_csv('data.csv'))
# เราใช้ df ดิบ ๆ จาก Section 2
np.random.seed(42)
n = 500
fuel_prices_base = {'ดีเซล': 44.0, 'แก๊สโซฮอล์ 95': 43.5, 'แก๊สโซฮอล์ 91': 42.8, 'เบนซิน': 50.2}

raw_df = pd.DataFrame({
    'date': pd.to_datetime(pd.date_range('2025-10-01', periods=n, freq='D')[np.random.permutation(n)]),
    'station': np.random.choice(['ปตท.', 'บางจาก', 'เชลล์', 'คาลเท็กซ์'], n),
    'fuel_type': np.random.choice(list(fuel_prices_base.keys()), n, p=[0.45, 0.30, 0.15, 0.10]),
    'car_type': np.random.choice(['รถเก๋ง', 'รถกระบะ', 'รถตู้', 'รถบรรทุก'], n, p=[0.4, 0.35, 0.15, 0.10]),
    'liters': np.round(np.random.uniform(20, 60, n), 2),
    'km_driven': np.round(np.random.uniform(300, 700, n)).astype(int),
})
raw_df['price_per_liter'] = raw_df['fuel_type'].map(fuel_prices_base) + np.random.normal(0, 0.5, n)
raw_df['total_cost'] = (raw_df['liters'] * raw_df['price_per_liter'] + np.random.normal(0, 10, n)).round(2)

# ใส่ความสกปรกเล็กน้อย
raw_df.loc[np.random.choice(n, 20, replace=False), 'liters'] = np.nan

print(f"STEP 1: โหลดข้อมูลดิบ {len(raw_df)} แถว")
raw_df.head(3)

In [ ]:
# ========== STEP 2: Data Cleaning ==========
clean_df = raw_df.copy()

# 2.1 เติม missing
clean_df['liters'] = clean_df['liters'].fillna(clean_df['liters'].median())

# 2.2 ลบ duplicate (ไม่มีในข้อมูลนี้ แต่ใส่ไว้กันเหนียว)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

# 2.3 ตัด outlier (ใช้ IQR)
Q1, Q3 = clean_df['liters'].quantile([0.25, 0.75])
IQR = Q3 - Q1
clean_df = clean_df[
    (clean_df['liters'] >= Q1 - 1.5*IQR) &
    (clean_df['liters'] <= Q3 + 1.5*IQR)
].reset_index(drop=True)

print(f"STEP 2: ทำความสะอาดเสร็จ เหลือ {len(clean_df)} แถว")

In [ ]:
# ========== STEP 3: Feature Engineering ==========
feat_df = clean_df.copy()

# 3.1 Feature จากวันที่
feat_df['day_of_week'] = feat_df['date'].dt.dayofweek
feat_df['month'] = feat_df['date'].dt.month
feat_df['is_weekend'] = (feat_df['day_of_week'] >= 5).astype(int)

# 3.2 Feature จากการคำนวณ
feat_df['km_per_liter'] = feat_df['km_driven'] / feat_df['liters']

# 3.3 One-Hot Encoding
feat_df = pd.get_dummies(feat_df, columns=['fuel_type', 'car_type', 'station'], drop_first=True)
# แปลง bool -> int
for col in feat_df.select_dtypes('bool').columns:
    feat_df[col] = feat_df[col].astype(int)

# 3.4 ตัดคอลัมน์ที่ไม่ใช้
feat_df = feat_df.drop(columns=['date'])

print(f"STEP 3: Feature Engineering เสร็จ มี {feat_df.shape[1]} คอลัมน์")
print(f"คอลัมน์: {list(feat_df.columns)[:8]}...")

In [ ]:
# ========== STEP 4: Split X, y และ Train/Test ==========
# y = สิ่งที่จะทำนาย = ค่าใช้จ่ายน้ำมัน
# X = feature ทั้งหมด

y = feat_df['total_cost']
X = feat_df.drop(columns=['total_cost'])

# Split ก่อน scale! (ป้องกัน Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"STEP 4: Split ข้อมูล")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")

In [ ]:
# ========== STEP 5: Feature Scaling (ป้องกัน Data Leakage!) ==========
# เลือกเฉพาะคอลัมน์ที่เป็นตัวเลขต่อเนื่อง (ไม่รวม one-hot 0/1)
numeric_cols = ['liters', 'price_per_liter', 'km_driven', 'km_per_liter', 'month']

scaler = StandardScaler()

# fit บน train เท่านั้น!
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# apply บน test (ใช้ mean, std ของ train)
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(f"STEP 5: Scale เสร็จ")
print(f"\nสถิติของ X_train หลัง scale (ควรใกล้ mean=0, std=1):")
print(X_train[numeric_cols].describe().round(2).loc[['mean', 'std']])

In [ ]:
# ========== STEP 6: พร้อมใช้งาน! ==========
print("=" * 50)
print("Data Prep Pipeline เสร็จสมบูรณ์!")
print("=" * 50)
print(f"\nX_train ready: {X_train.shape}")
print(f"X_test ready: {X_test.shape}")
print(f"y_train ready: {y_train.shape}")
print(f"y_test ready: {y_test.shape}")
print("\nสัปดาห์หน้า (Week 5) เราจะเอาข้อมูลชุดนี้ไป train ML Model จริง!")
print("-> Linear Regression, Decision Tree, Random Forest")

---
## สรุป Week 4
---

### Flow ที่เรียนวันนี้:

```
[ช่วง 1] ทบทวน + Math ปิดท้าย
Derivative = ความชัน -> Gradient มาจากไหน
Loss Landscape 3D -> AI เดินหาคำตอบ
Matrix -> ทำนายเยอะ ๆ ใน 1 บรรทัด
|
[ช่วง 2] เปิดข้อมูลในชีวิตจริง (ข้อมูลสกปรก!)
|
[ช่วง 3] Data Cleaning
Fix Type -> Remove Duplicates -> Fill Missing -> Outlier -> Normalize Text
|
[ช่วง 4] Feature Engineering
Date Features -> Derived Features -> One-Hot -> Binning -> Scaling
|
[ช่วง 5] กับดัก 3 ตัว
Bias / Noise / Data Leakage
|
[ช่วง 6] Full Pipeline -> พร้อมเข้า Week 5!
```

### คำศัพท์สำคัญ

| คำศัพท์ | ความหมาย |
|---------|----------|
| Derivative | ความชันของเส้น ณ จุดหนึ่ง |
| Loss Landscape | "หุบเขา" ของ loss ที่ AI ต้องเดินลง |
| Matrix Multiplication | `X @ W` - ทำนายหลายจุดพร้อมกัน |
| Missing Data | ข้อมูลว่าง (NaN) - เติมด้วย mean/median/mode |
| Outlier | ค่าผิดปกติ - ตรวจด้วย IQR |
| Feature Engineering | สร้าง feature ใหม่ให้ AI ฉลาดขึ้น |
| One-Hot Encoding | แปลง category -> คอลัมน์ 0/1 |
| Normalization | Min-Max -> [0, 1] |
| Standardization | Z-score -> mean=0, std=1 |
| Bias | ข้อมูลเอียง (systematic error) |
| Noise | ข้อมูลสกปรก (random error) |
| **Data Leakage** | **ข้อมูล test/อนาคตรั่วเข้า train** |

---
## Cheat Sheet: คำสั่งที่ใช้บ่อย
---

### Data Cleaning
```python
# Missing
df.isnull().sum()                         # นับ missing
df['col'].fillna(df['col'].median())      # เติม median
df.dropna()                               # ลบแถวที่มี missing

# Duplicate
df.duplicated().sum()                     # นับซ้ำ
df.drop_duplicates()                      # ลบซ้ำ

# Type
df['col'].astype(float)                   # แปลงเป็น float
df['col'].str.replace(' บาท', '')         # ลบข้อความ

# Outlier (IQR)
Q1, Q3 = df['col'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df = df[(df['col'] >= Q1-1.5*IQR) & (df['col'] <= Q3+1.5*IQR)]
```

### Feature Engineering
```python
# Date features
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# One-Hot
df = pd.get_dummies(df, columns=['category_col'])

# Binning
df['bin'] = pd.cut(df['num_col'], bins=[0, 10, 20, 30], labels=['low','mid','high'])

# Map text
df['col'] = df['col'].replace({'Diesel': 'ดีเซล'})
```

### Scaling (ต้อง split ก่อน!)
```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 1. Split ก่อน
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 2. Fit บน train เท่านั้น
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 3. Apply บน test
X_test_scaled = scaler.transform(X_test)
```

### Math for ML (SymPy)
```python
import sympy as sp

x = sp.Symbol('x')
f = x**2 - 4*x + 5
f_prime = sp.diff(f, x)                   # หา derivative
grad_func = sp.lambdify(x, f_prime)       # เปลี่ยนเป็น function
```

---
## การบ้านสัปดาห์นี้
---

1. **ทำ Workshop ให้ครบ** — Workshop 1 (Math) + Workshop 2 (Data Cleaning) ถ้ายังไม่เสร็จ

2. **ดาวน์โหลด Dataset จริงจาก Kaggle** (เช่น `Fuel Consumption`, `Car Price Dataset`) แล้วทำ:
   - Clean ข้อมูล (missing, duplicate, outlier)
   - สร้าง feature ใหม่อย่างน้อย 2 ตัว
   - One-Hot Encoding
   - Train/Test Split แล้ว Scale (ระวัง Data Leakage!)

3. **(Bonus)** ลองเปลี่ยนวิธีเติม missing data จาก `median` เป็น:
   - `mean`
   - `groupby().transform(median)`
   - สังเกตว่าผลต่างกันยังไง

4. ส่ง link Colab Notebook ในกลุ่ม Facebook

---

**สัปดาห์หน้า (Week 5):** Supervised Learning — Regression & Classification
- Linear Regression (ใช้ sklearn จริง!)
- Logistic Regression
- Classification: ทำนาย Yes/No
- เอาข้อมูลที่เตรียมไว้ Week 4 มา Train Model จริง ๆ!